# 30 后训练如何用 KL 控制和发布回滚防止策略漂移？

## 面试回答主线

后训练不能只追 reward：策略若快速远离 reference，常会出现格式退化、安全边界变化或 reward overfitting。KL controller 根据观测 KL 与目标 KL 自适应调节惩罚系数；发布层还需把 reward、KL、verifier、安全和延迟放进门禁，并能回滚到最后健康 checkpoint。实验用六个客服策略 checkpoint 的 reward、KL 和安全分数，手写自适应 beta 更新与健康窗口选择；故意设置一个 reward 很高但 KL/安全越界的 checkpoint，展示它被拒绝并回滚。

**核心公式：** 目标可写为 $J=R-\beta\operatorname{KL}(\pi_\theta\|\pi_{ref})$；简单 controller 可令 $\beta\leftarrow\beta\exp(\alpha(\widehat{KL}/KL_{target}-1))$。发布需同时满足 $KL\le K$、安全/验证阈值与回归测试。

后续依次展示同数据基线、手写核心状态/概率、结果表、真实失败与修复。数值仅用于机制验证。


## 真实案例

数据是六条脱敏客服 prompt，每条含 chosen/rejected 回答；注意力主题会将它们映射成流式键值事件。字段语义和失败模式与真实系统一致，但样本规模不能代表线上效果。


In [1]:
import math  # 导入数学函数实现概率和复杂度公式。
import warnings  # 导入警告控制模块保持输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖产生的弃用提示。
import torch  # 导入张量和自动微分基础能力。
import torch.nn as nn  # 导入模块基类以显式定义网络。
torch.manual_seed(41)  # 固定随机种子保证输出可复现。
torch.set_num_threads(1)  # 固定小实验 CPU 线程数。
samples = [  # 定义六条可读的 prompt、候选回复或流式事件。
    {'id': 'P01', 'prompt': '支付重复扣款怎么处理？', 'chosen': '核验订单后原路退款。', 'rejected': '无需核验直接忽略。'},  # 退款决策样本。
    {'id': 'P02', 'prompt': '发现陌生转账怎么办？', 'chosen': '立即冻结并核验身份。', 'rejected': '等待下个账单周期。'},  # 账户安全样本。
    {'id': 'P03', 'prompt': '收不到登录验证码？', 'chosen': '检查手机号并重发。', 'rejected': '建议注销账户。'},  # 登录支持样本。
    {'id': 'P04', 'prompt': '地址如何修改？', 'chosen': '在发货前更新地址。', 'rejected': '永久不可修改。'},  # 售后样本。
    {'id': 'P05', 'prompt': '银行卡被盗刷？', 'chosen': '冻结卡并保留证据。', 'rejected': '继续正常使用。'},  # 风险样本。
    {'id': 'P06', 'prompt': '发票抬头写错？', 'chosen': '按规则更正抬头。', 'rejected': '删除全部订单。'},  # 账单样本。
]  # 结束可读数据定义。
print('教学实验：六条脱敏客服 prompt/候选或流式状态，只解释训练与状态机制。')  # 声明实验边界。
for row in samples:  # 逐条展示 prompt/chosen/rejected。
    print(f"{row['id']} | 问题={row['prompt']} | chosen={row['chosen']} | rejected={row['rejected']}")  # 输出真实语义样本。


教学实验：六条脱敏客服 prompt/候选或流式状态，只解释训练与状态机制。
P01 | 问题=支付重复扣款怎么处理？ | chosen=核验订单后原路退款。 | rejected=无需核验直接忽略。
P02 | 问题=发现陌生转账怎么办？ | chosen=立即冻结并核验身份。 | rejected=等待下个账单周期。
P03 | 问题=收不到登录验证码？ | chosen=检查手机号并重发。 | rejected=建议注销账户。
P04 | 问题=地址如何修改？ | chosen=在发货前更新地址。 | rejected=永久不可修改。
P05 | 问题=银行卡被盗刷？ | chosen=冻结卡并保留证据。 | rejected=继续正常使用。
P06 | 问题=发票抬头写错？ | chosen=按规则更正抬头。 | rejected=删除全部订单。


## Baseline / 基线

先运行最朴素、但同样使用这些输入和同一指标的对照，避免只看一个核心算法数字。


In [2]:
checkpoints = [{'name': 'ckpt-01', 'reward': 0.62, 'kl': 0.05, 'safety': 0.99}, {'name': 'ckpt-02', 'reward': 0.68, 'kl': 0.08, 'safety': 0.99}, {'name': 'ckpt-03', 'reward': 0.73, 'kl': 0.11, 'safety': 0.98}, {'name': 'ckpt-04', 'reward': 0.91, 'kl': 0.42, 'safety': 0.71}, {'name': 'ckpt-05', 'reward': 0.76, 'kl': 0.12, 'safety': 0.98}, {'name': 'ckpt-06', 'reward': 0.78, 'kl': 0.16, 'safety': 0.97}]  # 定义六个后训练 checkpoint 的 reward、KL 与安全评测。
reward_best = max(checkpoints, key=lambda row: row['reward'])  # 错误地只按 reward 选最高 checkpoint。
baseline_metric = reward_best['reward']  # 保存纯 reward 选择的分数。
print(f'仅按 reward 会选择 {reward_best["name"]}：reward={reward_best["reward"]:.2f}，KL={reward_best["kl"]:.2f}，safety={reward_best["safety"]:.2f}')  # 暴露高 reward checkpoint 的风险。


仅按 reward 会选择 ckpt-04：reward=0.91，KL=0.42，safety=0.71


## 手写核心实现与中间量

核心实现保留 state、ratio、优势、mask 或概率分母等中间量，不用 Trainer 或现成 Agent/Attention 框架遮蔽机制。


In [3]:
target_kl = 0.12  # 设置训练期目标 KL。
beta = 0.10  # 初始化 KL 惩罚系数。
beta_trace = []  # 保存各 checkpoint 后的自适应 beta。
for row in checkpoints:  # 逐 checkpoint 根据观测 KL 调整 beta。
    beta *= math.exp(0.35 * (row['kl'] / target_kl - 1.0))  # 手写指数型 KL controller。
    beta = min(1.0, max(0.01, beta))  # 对 beta 加上下限避免极端冻结或失控。
    beta_trace.append(beta)  # 保存 controller 状态。
healthy = [row for row in checkpoints if row['kl'] <= 0.18 and row['safety'] >= 0.95]  # 定义 KL 与安全双门禁。
released = max(healthy, key=lambda row: row['reward'])  # 在健康窗口中选择 reward 最优版本。
core_metric = released['reward']  # 保存实际发布版本 reward。
print(f'beta 轨迹={ [round(value, 3) for value in beta_trace] }，健康候选={[row["name"] for row in healthy]}，发布={released["name"]}')  # 输出控制器和发布决策。


beta 轨迹=[0.082, 0.073, 0.07, 0.169, 0.169, 0.19]，健康候选=['ckpt-01', 'ckpt-02', 'ckpt-03', 'ckpt-05', 'ckpt-06']，发布=ckpt-06


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 建立基线与核心的同口径结果表。
for name, metric in comparison_rows:  # 逐行输出结果表。
    print(f'{name:<8} | 指标={metric:.6f}')  # 显示可读数值对照。


Baseline | 指标=0.910000
核心机制     | 指标=0.780000


## 结果解读

这里只能得出本受控样本上的机制结论。生产应以 token/样本加权 KL、分 prompt 类别、影子流量和分阶段 rollout 做控制；beta 调整需要限幅，避免一次异常批把训练冻结。 生产决策必须进一步看验证集、线上安全指标、算力和版本可追溯性。

## 失败案例

下方先让关键条件真实失效，再展示修复如何改变可观测指标。


In [5]:
failure_metric = int(reward_best['name'] == 'ckpt-04')  # 标记高 reward 但越界 checkpoint 被错误选中的事实。
fix_metric = released['name']  # 记录门禁后实际发布的健康 checkpoint。
rollback_target = max([row for row in checkpoints if row['name'] < 'ckpt-04' and row['kl'] <= 0.18 and row['safety'] >= 0.95], key=lambda row: row['reward'])  # 找到事故前最后健康版本。
print(f'失败：{reward_best["name"]} reward 高但 KL/安全越界；修复：发布 {fix_metric}，若事故发生回滚到 {rollback_target["name"]}')  # 展示真正可执行的回滚目标。


失败：ckpt-04 reward 高但 KL/安全越界；修复：发布 ckpt-06，若事故发生回滚到 ckpt-03


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产应以 token/样本加权 KL、分 prompt 类别、影子流量和分阶段 rollout 做控制；beta 调整需要限幅，避免一次异常批把训练冻结。

**常见坑：** 只因 reward 最高就发布，或只看平均 KL 忽略长尾 prompt；没有权威状态与版本 manifest 的“回滚”只是口头承诺。

**延伸追问：** reverse KL 与 forward KL 在后训练中有什么不同？如何设置安全事故触发的自动停止与人工审批边界？

## 生产差距

实验运行于 CPU/FP32，只有 6 条离线样本，省略了真实 rollout、分布式同步、混合精度、内容安全、数据治理、checkpoint 和监控。上线版本应以受审计的状态、指标和回滚流程替代这些教学变量。


In [6]:
assert failure_metric == 1  # 验证纯 reward 策略确实会误选风险 checkpoint。
assert released['name'] != reward_best['name']  # 验证多指标门禁拒绝了风险版本。
assert released['kl'] <= 0.18  # 验证发布版本满足 KL 上限。
assert released['safety'] >= 0.95  # 验证发布版本满足安全门槛。
